In [14]:
# Setup: imports and configuration
import os
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
import mlflow

from neurodecoders.data.loading import load_npz_dataset, normalize_images_and_rates
from neurodecoders.mlflow_utils.utils import create_firing_rate_scatterplot

# Point MLflow to the local tracking dir (same as encoder.ipynb)
path = "/Users/laura/source/github/lauraporta/neurodecoders/mlruns"
mlflow.set_tracking_uri(path)

# Torch device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [19]:
# List experiments (minimal)
experiments = mlflow.search_experiments()

for e in experiments:
    print(e.experiment_id, e.name, e.artifact_location)

855673514087340178 my_experiment file:///Users/laura/source/github/lauraporta/neurodecoders/mlruns/855673514087340178
493472811057638535 neurodecoders file:///Users/laura/source/github/lauraporta/neurodecoders/mlruns/493472811057638535
159415702772134731 test_decoder_plots file:///Users/laura/source/github/lauraporta/neurodecoders/mlruns/159415702772134731
0 Default file:///Users/laura/source/github/lauraporta/neurodecoders/mlruns/0


In [16]:
# Quick check of tracking URI and experiments
print("tracking_uri:", mlflow.get_tracking_uri())
for e in experiments:
    print(e.experiment_id, e.name, e.artifact_location)

# Count runs across ALL experiments
all_exp_ids = [e.experiment_id for e in experiments]
all_runs = mlflow.search_runs(all_exp_ids, max_results=5)
print("sample runs shape:", all_runs.shape)

# If this is not the MLflow you expect, set the correct one here and re-run cells:
# mlflow.set_tracking_uri("file:///ABSOLUTE/PATH/TO/mlruns")
# mlflow.set_tracking_uri("http://localhost:5000")


tracking_uri: /Users/laura/source/github/lauraporta/neurodecoders/mlruns
855673514087340178 my_experiment file:///Users/laura/source/github/lauraporta/neurodecoders/mlruns/855673514087340178
493472811057638535 neurodecoders file:///Users/laura/source/github/lauraporta/neurodecoders/mlruns/493472811057638535
159415702772134731 test_decoder_plots file:///Users/laura/source/github/lauraporta/neurodecoders/mlruns/159415702772134731
0 Default file:///Users/laura/source/github/lauraporta/neurodecoders/mlruns/0
sample runs shape: (5, 70)


In [17]:
# Pick experiment and list runs with proper MLflow columns and model discovery
EXPERIMENT_NAME = "neurodecoders"  # set to a string to filter
exp_ids = [e.experiment_id for e in experiments if (EXPERIMENT_NAME is None or e.name == EXPERIMENT_NAME)]

# Query runs
_df = mlflow.search_runs(exp_ids, order_by=["attributes.start_time DESC"], max_results=50)

# Build columns list safely. In MLflow, run name is stored in the tag 'mlflow.runName'
base_cols = ["run_id", "experiment_id", "status", "artifact_uri", "start_time", "end_time"]
name_col = "tags.mlflow.runName" if "tags.mlflow.runName" in _df.columns else None
cols = base_cols + ([name_col] if name_col else [])

runs_df = _df[cols].copy()
if name_col:
    runs_df = runs_df.rename(columns={name_col: "run_name"})

# Map experiment_id -> experiment_name for readability
exp_id_to_name = {e.experiment_id: e.name for e in experiments}
runs_df["experiment_name"] = runs_df["experiment_id"].map(exp_id_to_name)

# Detect top-level model artifact directory by scanning for MLflow model dirs (containing 'MLmodel')
from mlflow.tracking import MlflowClient
_client = MlflowClient()

def _detect_model_dir(run_id: str) -> str | None:
    try:
        for art in _client.list_artifacts(run_id):
            if art.is_dir:
                # Look for an MLflow model dir (contains 'MLmodel')
                children = _client.list_artifacts(run_id, art.path)
                if any(child.path.endswith("MLmodel") for child in children):
                    return art.path
        return None
    except Exception:
        return None

runs_df["model_artifact"] = runs_df["run_id"].apply(_detect_model_dir)

# Reorder for display
display_cols = [
    "run_id",
    "run_name" if "run_name" in runs_df.columns else None,
    "experiment_id",
    "experiment_name",
    "status",
    "artifact_uri",
    "model_artifact",
    "start_time",
    "end_time",
]
runs_df = runs_df[[c for c in display_cols if c is not None]]
runs_df

,run_id,run_name,experiment_id,experiment_name,status,artifact_uri,model_artifact,start_time,end_time
0,1912ce9cca5d47f9a6ed05cf8a171ecb,rumbling-gnu-830,493472811057638535,neurodecoders,FINISHED,file:///Users/laura/source/github/lauraporta/n...,None,2025-09-22 14:11:07.582000+00:00,2025-09-22 14:11:29.114000+00:00
1,3fac5aeb9638451bbf67281e1467af94,tasteful-zebra-600,493472811057638535,neurodecoders,FINISHED,file:///Users/laura/source/github/lauraporta/n...,None,2025-09-22 14:02:39.799000+00:00,2025-09-22 14:03:05.766000+00:00
2,a70a7282f337418182bbbd6ce82130ee,spiffy-owl-473,493472811057638535,neurodecoders,FAILED,file:///Users/laura/source/github/lauraporta/n...,None,2025-09-22 13:27:15.899000+00:00,2025-09-22 13:27:15.913000+00:00


In [10]:
# Pick a run id (minimal). Optionally set RUN_ID manually.
RUN_ID = None

if RUN_ID is None:
    # Just take the most recent row
    if len(runs_df) == 0:
        raise RuntimeError("No runs found. Check tracking URI and experiment filter.")
    RUN_ID = runs_df.iloc[0]["run_id"]

print(RUN_ID)


1912ce9cca5d47f9a6ed05cf8a171ecb


In [11]:
# Load encoder model (minimal)
import mlflow.pytorch

for art in ["encoder_model", "model", "trained_model"]:
    try:
        uri = f"runs:/{RUN_ID}/{art}"
        encoder_model = mlflow.pytorch.load_model(uri)
        print(uri)
        break
    except Exception:
        encoder_model = None

if encoder_model is None:
    raise RuntimeError("No model artifact found in run")

encoder_model = encoder_model.to(device)
encoder_model.eval()
encoder_model

RuntimeError: No model artifact found in run

In [8]:
# Load the latest available test dataset from workspace
from glob import glob

synthetic_test_dir = Path(get_path("workspace/datasets/synthetic")) / "test"
npz_files = sorted(glob(str(synthetic_test_dir / "*.npz")))
if not npz_files:
    raise FileNotFoundError(f"No test .npz found under {synthetic_test_dir}")

# Use the most recent by mtime
latest_npz = max(npz_files, key=os.path.getmtime)
print(f"Using test dataset: {latest_npz}")

images, firing_true = load_npz_dataset(latest_npz)
images_norm, firing_true_norm, H, W = normalize_images_and_rates(images, firing_true)

# Ensure shape (N, C, H, W) with C=1 if grayscale
if images_norm.ndim == 3:
    images_norm = images_norm[:, None, :, :]

images_norm.shape, firing_true_norm.shape


FileNotFoundError: No test .npz found under workspace/datasets/synthetic/test

In [9]:
# Inference on test set
batch_size = 64

X_tensor = torch.from_numpy(images_norm.astype(np.float32))
Y_true_tensor = torch.from_numpy(firing_true_norm.astype(np.float32))

dataloader = DataLoader(TensorDataset(X_tensor, Y_true_tensor), batch_size=batch_size, shuffle=False)

all_preds = []
with torch.no_grad():
    for xb, yb in dataloader:
        xb = xb.to(device)
        preds = encoder_model(xb)
        all_preds.append(preds.detach().cpu().numpy())

Y_pred = np.concatenate(all_preds, axis=0)
Y_true = firing_true_norm
Y_pred.shape, Y_true.shape


NameError: name 'images_norm' is not defined

In [10]:
# Reproduce the final verification plot: means and stds scatter
true_mean = Y_true.mean(axis=0)
pred_mean = Y_pred.mean(axis=0)
true_std = Y_true.std(axis=0)
pred_std = Y_pred.std(axis=0)

plots_dir = Path(get_path("workspace/plots"))
plots_dir.mkdir(parents=True, exist_ok=True)
plot_path = plots_dir / "firing_rate_analysis_inference.png"

_ = create_firing_rate_scatterplot(
    true_mean=true_mean,
    pred_mean=pred_mean,
    true_std=true_std,
    pred_std=pred_std,
    save_path=str(plot_path),
    title="Firing Rate Analysis (Inference on Test)",
)

from IPython.display import Image, display
display(Image(filename=str(plot_path)))


NameError: name 'Y_true' is not defined

In [11]:
# Optional: compute simple correlation across neurons as in verification
# Pearson correlation per neuron between predicted and true rates
from scipy.stats import pearsonr

corrs = []
for i in range(Y_true.shape[1]):
    c = pearsonr(Y_true[:, i], Y_pred[:, i])[0]
    if np.isfinite(c):
        corrs.append(c)

print(f"Mean test correlation: {np.mean(corrs):.4f} ± {np.std(corrs):.4f} (n={len(corrs)})")


NameError: name 'Y_true' is not defined